# الفصل 19: تدريب Tokenizer مخصص

> طبقة الحدس في الجزء الثاني (الفصل 6) تعلّمنا آلية BPE وWordPiece وSentencePiece من الصفر.

- **الجزء:** 5 — بناء LLM من الصفر
- **الفصل:** 19
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 18: مصادر البيانات وتنظيفها
- **الوسوم:** Fertility، Arabic NLP، SentencePiece، Vocabulary

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** tokenizers
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("tokenizers", "tokenizers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch19-custom-tokenizer.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس في الجزء الثاني (الفصل 6) تعلّمنا آلية BPE وWordPiece وSentencePiece من الصفر.

> **بيانات الفصل**
>
> - **الجزء:** 5 — بناء LLM من الصفر
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 18: مصادر البيانات وتنظيفها


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم لماذا التطبيع (Normalization) حرج جدًا للعربية؟
- فهم معالجات تطبيع إضافية خاصة بالعربية
- فهم تدريب Tokenizer فعلي باستخدام مكتبة `tokenizers` من Hugging Face


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**

في الجزء الثاني (الفصل 6) تعلّمنا آلية BPE وWordPiece وSentencePiece من الصفر. لكن هناك فرق جوهري بين "فهم آلية Tokenizer" و"تدريب Tokenizer مُحسَّن فعليًا لمشروعك الخاص". إذا كنت تبني LLM يستهدف اللغة العربية بشكل أساسي، فاستخدام Tokenizer جاهز مُدرَّب أساسًا على الإنجليزية (مثل GPT-2 tokenizer الأصلي) سيكون **كارثيًا من ناحية الكفاءة**: كل حرف عربي أو مجموعة أحرف قد تتحلل إلى عدة Bytes/Tokens بدلاً من رمز واحد مُعبِّر، مما يُضاعف طول التسلسل لنفس المحتوى، ويُضاعف تكلفة الحوسبة والذاكرة (KV Cache) بلا أي داعٍ.

المقياس الذي نستخدمه لقياس "جودة" Tokenizer للغة معينة هو **Fertility** (الخصوبة):

$$\text{Fertility} = \frac{\text{عدد الـ Tokens الناتجة}}{\text{عدد الكلمات الأصلية}}$$

كلما اقترب هذا الرقم من 1.0-1.5 لنص معين، كان الـ Tokenizer أكثر كفاءة لتلك اللغة. القيم المرتفعة (3، 4، أو أكثر) تعني أن كل كلمة تتفتت لعدة رموز، مما يُهدر طول سياق النموذج (Context Window) ويزيد التكلفة الحسابية بشكل مباشر.


## 2. الخريطة الذهنية


```text
تدريب Tokenizer مخصص
│
├── 1. اختيار الخوارزمية (BPE بايتي / Unigram / WordPiece)
│
├── 2. اختيار حجم المفردات (Vocabulary Size)
│     32K؟ 64K؟ 128K؟ تريد-أوف بين الكفاءة والذاكرة
│
├── 3. التطبيع المسبق (Pre-tokenization Normalization)
│     NFC/NFKC للعربية، معالجة التشكيل، الأرقام
│
├── 4. التدريب الفعلي على عيّنة تمثيلية من بيانات الهدف
│
├── 5. قياس الجودة (Fertility, Compression Ratio)
│
└── 6. دمج رموز خاصة (Special Tokens: <|endoftext|>, <|user|>, <|assistant|>...)
```


## 3. الشرح المتدرج

### 3.1 لماذا التطبيع (Normalization) حرج جدًا للعربية؟

> 🟡 **طبقة الآلية**

النصوص اليونيكود (Unicode) يمكن تمثيلها بأكثر من طريقة بايتية مع نفس الشكل المرئي. مثال كلاسيكي بالعربية: الحرف "أ" (ألف بهمزة فوق) يمكن تمثيله إما:
1. كنقطة كود واحدة مُركَّبة مسبقًا (Precomposed): `U+0623`
2. أو كتسلسل من نقطتين منفصلتين (Decomposed): الألف `U+0627` + علامة الهمزة فوق `U+0654`

إذا احتوت بيانات تدريبك على مزيج من الشكلين لنفس الكلمة (وهذا شائع جدًا في بيانات الويب العربية لأنها تأتي من مصادر ومحررات نصوص مختلفة)، فإن Tokenizer **سيتعامل معهما كرمزين مختلفين تمامًا**، رغم أنهما بصريًا ودلاليًا نفس الحرف! هذا يُضعِف كفاءة الترميز (نحتاج تمثيل كلا الشكلين في المفردات) ويُربك النموذج لاحقًا (نفس الكلمة لها تمثيلان مختلفان في الفضاء الشعاعي).

الحل المعياري هو **تطبيع NFC** (Normalization Form C — Canonical Composition): تحويل كل النصوص لصيغة مُركَّبة موحَّدة قبل أي معالجة أخرى.


In [ ]:
import unicodedata

def normalize_arabic_text_nfc(text: str) -> str:
    """
    تطبيع NFC: يحوّل كل التمثيلات البديلة لنفس الحرف المرئي
    إلى تمثيل Unicode قانوني (Canonical) موحّد.
    هذه الخطوة ضرورية قبل تدريب أي Tokenizer على نصوص عربية
    لتجنب معاملة نفس الحرف كرمزين مختلفين في المفردات.
    """
    return unicodedata.normalize("NFC", text)

def demonstrate_normalization_problem():
    """توضيح عملي لمشكلة عدم التطبيع"""
    # نفس الكلمة "أحمد" لكن بتمثيلين مختلفين بايتيًا
    word_precomposed = "\u0623\u062d\u0645\u062f"  # أحمد (الألف مع الهمزة مُركَّبة)
    word_decomposed = "\u0627\u0654\u062d\u0645\u062f"  # أحمد (الألف ثم الهمزة منفصلة)

    print(f"الشكل المرئي للكلمة الأولى: {word_precomposed}")
    print(f"الشكل المرئي للكلمة الثانية: {word_decomposed}")
    print(f"هل هما متطابقتان بايتيًا قبل التطبيع؟ "
          f"{word_precomposed == word_decomposed}")  # False!

    normalized_1 = normalize_arabic_text_nfc(word_precomposed)
    normalized_2 = normalize_arabic_text_nfc(word_decomposed)
    print(f"هل هما متطابقتان بعد تطبيع NFC؟ "
          f"{normalized_1 == normalized_2}")  # True

demonstrate_normalization_problem()


**الناتج**:


```text
الشكل المرئي للكلمة الأولى: أحمد
الشكل المرئي للكلمة الثانية: أحمد
هل هما متطابقتان بايتيًا قبل التطبيع؟ False
هل هما متطابقتان بعد تطبيع NFC؟ True
```


### 3.2 معالجات تطبيع إضافية خاصة بالعربية


In [ ]:
import re

class ArabicTextNormalizer:
    """
    خط أنابيب تطبيع شامل للنصوص العربية قبل تدريب Tokenizer.
    يجب تطبيق هذه الخطوات بثبات على كل من بيانات تدريب Tokenizer
    وبيانات تدريب النموذج لاحقًا (وإلا سيحدث عدم تطابق).
    """
    def __init__(self,
                 remove_diacritics: bool = False,
                 normalize_alef_forms: bool = False,
                 normalize_numerals: bool = True):
        """
        remove_diacritics: إزالة التشكيل (محل جدل! بعض المشاريع تحتفظ به
            للنصوص الدينية/الشعرية، لكن معظم LLMs العامة تزيله لتقليل
            حجم المفردات وزيادة الكفاءة).
        normalize_alef_forms: توحيد أ/إ/آ إلى ا (يُفقِد معلومة نحوية لكن
            يقلل تشتت المفردات؛ قرار هندسي يعتمد على حالة الاستخدام).
        normalize_numerals: توحيد الأرقام العربية-الهندية (٠١٢)
            إلى أرقام عربية-غربية (012) لتوحيد التمثيل.
        """
        self.remove_diacritics = remove_diacritics
        self.normalize_alef_forms = normalize_alef_forms
        self.normalize_numerals = normalize_numerals

        self.diacritics_pattern = re.compile(r'[\u064B-\u0652\u0670\u0640]')
        self.alef_forms_pattern = re.compile(r'[إأآا]')
        self.eastern_to_western_digits = str.maketrans(
            "٠١٢٣٤٥٦٧٨٩", "0123456789"
        )

    def normalize(self, text: str) -> str:
        # الخطوة 1: تطبيع NFC أولاً وقبل كل شيء (إلزامي دائمًا)
        text = unicodedata.normalize("NFC", text)

        # الخطوة 2: إزالة التشكيل (اختياري)
        if self.remove_diacritics:
            text = self.diacritics_pattern.sub('', text)

        # الخطوة 3: توحيد أشكال الألف (اختياري، قرار هندسي)
        if self.normalize_alef_forms:
            text = self.alef_forms_pattern.sub('ا', text)

        # الخطوة 4: توحيد الأرقام
        if self.normalize_numerals:
            text = text.translate(self.eastern_to_western_digits)

        # الخطوة 5: توحيد المسافات
        text = re.sub(r'\s+', ' ', text).strip()
        return text


> ⚠️ **خطأ شائع**: توحيد أشكال الألف أو إزالة التشكيل قرار **لا يمكن الرجوع عنه** بسهولة؛ إن قررت إزالة التشكيل من بيانات التدريب فلن يتمكن النموذج أبدًا من توليد نص مُشكَّل بشكل صحيح (لأنه لم يرَ أي مثال عليه). إن كان مشروعك يستهدف الشعر العربي الكلاسيكي أو النصوص الدينية حيث التشكيل أساسي، فيجب الاحتفاظ به.

### 3.3 تدريب Tokenizer فعلي باستخدام مكتبة `tokenizers` من Hugging Face


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from tokenizers.normalizers import NFC, Sequence
from tokenizers.processors import TemplateProcessing

def train_bpe_tokenizer_for_arabic(
    corpus_files: list,
    vocab_size: int = 50_000,
    min_frequency: int = 2,
    special_tokens: list = None,
) -> Tokenizer:
    """
    تدريب Tokenizer من نوع Byte-Level BPE مُحسَّن للنصوص العربية،
    بنفس المنهجية المستخدمة في تدريب tokenizers نماذج مثل GPT وLLaMA،
    لكن مع تطبيع NFC مُدمَج في خط الأنابيب لضمان الكفاءة مع العربية.
    """
    special_tokens = special_tokens or [
        "<|endoftext|>",   # نهاية الوثيقة
        "<|pad|>",         # الحشو
        "<|unk|>",         # غير معروف (نادر الاستخدام في BPE البايتي)
        "<|user|>",        # بداية رسالة المستخدم (لمرحلة SFT لاحقًا)
        "<|assistant|>",   # بداية رد المساعد
        "<|system|>",      # رسالة النظام
    ]

    # 1. إنشاء Tokenizer بنموذج BPE خام
    tokenizer = Tokenizer(models.BPE(unk_token="<|unk|>"))

    # 2. تطبيع NFC كخطوة أولى إلزامية قبل أي تقطيع
    tokenizer.normalizer = Sequence([NFC()])

    # 3. Pre-tokenization بمستوى البايت (ByteLevel)
    # هذا يضمن عدم وجود رموز "unknown" إطلاقًا، لأن أي نص يمكن تفكيكه
    # لبايتات خام (256 قيمة ممكنة) حتى لو لم يظهر الحرف في بيانات التدريب
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = pre_tokenizers.ByteLevel.decoder if hasattr(
        pre_tokenizers.ByteLevel, 'decoder') else None

    # 4. إعداد المدرِّب (Trainer)
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=special_tokens,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # كل البايتات الـ256
        show_progress=True,
    )

    # 5. التدريب الفعلي من ملفات الكوربص
    tokenizer.train(files=corpus_files, trainer=trainer)

    # 6. إضافة معالج القالب (Template) لإضافة رموز البداية/النهاية تلقائيًا
    tokenizer.post_processor = TemplateProcessing(
        single="$A <|endoftext|>",
        special_tokens=[("<|endoftext|>",
                          tokenizer.token_to_id("<|endoftext|>"))],
    )

    return tokenizer

def evaluate_tokenizer_fertility(tokenizer: Tokenizer, test_texts: list) -> dict:
    """
    قياس Fertility: متوسط عدد الـ Tokens لكل كلمة عبر مجموعة اختبار.
    نحسبها أيضًا بمقارنة Tokenizer جديد بـ Tokenizer مرجعي (مثل GPT-2 الأصلي)
    لتوضيح الفارق الكمي في الكفاءة.
    """
    total_words = 0
    total_tokens = 0
    total_chars = 0

    for text in test_texts:
        words = text.split()
        encoded = tokenizer.encode(text)
        total_words += len(words)
        total_tokens += len(encoded.ids)
        total_chars += len(text)

    fertility = total_tokens / max(total_words, 1)
    compression_ratio = total_chars / max(total_tokens, 1)  # أحرف لكل رمز

    return {
        "total_words": total_words,
        "total_tokens": total_tokens,
        "fertility": fertility,                 # كلما قلّ كان أفضل
        "chars_per_token": compression_ratio,    # كلما زاد كان أفضل (ضغط أعلى)
    }


### 3.4 اختيار حجم المفردات (Vocabulary Size)

> 🔴 **طبقة التعمّق**

هذا قرار هندسي يتضمن تريد-أوف دقيق:

| حجم المفردات | الأثر على طول التسلسل | الأثر على حجم النموذج | الاستخدام النموذجي |
|---|---|---|---|
| صغير (8K-16K) | أطول (Fertility أعلى) | طبقة Embedding/Output أصغر | نماذج صغيرة جدًا، لغة واحدة بسيطة |
| متوسط (32K-50K) | متوازن | متوسط | معظم نماذج اللغة الواحدة (مثل GPT-2 الأصلي: 50K) |
| كبير (100K-128K) | أقصر (Fertility أقل) | طبقة Embedding/Output أكبر بكثير | نماذج متعددة اللغات (مثل LLaMA-3: 128K، GPT-4: ~100K) |
| ضخم جدًا (250K+) | قصير جدًا | يهيمن على عدد معاملات النموذج الصغيرة | نماذج ضخمة جدًا اللغات (مثل Gemini، NLLB) |

**لماذا يزيد حجم المفردات مع زيادة عدد اللغات المدعومة؟** لأن كل لغة تحتاج مجموعة فرعية كافية من الرموز لتمثيلها بكفاءة (Fertility منخفض)؛ فنموذج متعدد اللغات يحتاج "حصة" كافية من المفردات لكل لغة مستهدفة، وإلا ستعاني اللغات غير الإنجليزية (ومنها العربية) من Fertility مرتفع.

**الكلفة الحسابية لحجم المفردات**: عدد معاملات طبقتي Embedding والـ Output Head (في حال عدم ربطهما Weight Tying) يُحسب كالتالي:

$$\text{معاملات Embedding} = V \times d_{model}$$

مثال: لنموذج بـ $d_{model}=4096$ (حجم LLaMA-7B تقريبًا)، الفرق بين مفردات 32K و128K:
- عند V=32,000: المعاملات = 32,000 × 4,096 ≈ **131 مليون معامل**
- عند V=128,000: المعاملات = 128,000 × 4,096 ≈ **524 مليون معامل**

هذا فرق يقارب **400 مليون معامل إضافي** فقط في طبقة Embedding! لذلك اختيار حجم المفردات ليس قرارًا تافهًا، بل يؤثر مباشرة على حجم النموذج الكلي وتكلفة التدريب.


In [ ]:
def calculate_embedding_parameters_tradeoff(d_model: int,
                                               vocab_sizes: list,
                                               weight_tying: bool = True):
    """
    حساب كلفة معاملات Embedding/Output Head عند أحجام مفردات مختلفة،
    لمساعدة اتخاذ القرار الهندسي عند تصميم Tokenizer.
    """
    print(f"{'حجم المفردات':<15}{'معاملات Embedding':<22}{'معاملات Output':<20}{'الإجمالي':<15}")
    print("-" * 72)
    for vocab_size in vocab_sizes:
        embedding_params = vocab_size * d_model
        # إذا استخدمنا Weight Tying (ربط أوزان Embedding وOutput Head،
        # كما في GPT-2 وLLaMA)، لا نحسب معاملات Output بشكل منفصل
        output_params = 0 if weight_tying else vocab_size * d_model
        total = embedding_params + output_params
        print(f"{vocab_size:<15,}{embedding_params:<22,}{output_params:<20,}{total:<15,}")

calculate_embedding_parameters_tradeoff(
    d_model=4096,
    vocab_sizes=[32_000, 50_000, 100_000, 128_000],
    weight_tying=True
)


## 4. مثال محلول خطوة بخطوة

لنقارن Fertility بين Tokenizer إنجليزي قياسي (GPT-2) وTokenizer عربي مخصص على نفس الجملة العربية:

**الجملة الاختبارية**:
> "تساهم نماذج المحوّلات في تطوير تطبيقات معالجة اللغات الطبيعية بشكل كبير"

عدد الكلمات (تقسيم على المسافات) = **10 كلمات**.

**باستخدام GPT-2 Tokenizer الأصلي** (مُدرَّب بشكل شبه حصري على الإنجليزية):

نظرًا لأن GPT-2 BPE لم يرَ عربيًا كافيًا أثناء التدريب، فإن كل حرف عربي تقريبًا يتحلل إلى تسلسل بايتات UTF-8 منفصلة (كل حرف عربي = 2 بايت في UTF-8، وكل بايت قد يصبح رمزًا منفصلاً إن لم يتكرر النمط بما يكفي ليُدمَج):


```text
تقدير: ~10 كلمات → ~45-55 رمزًا (Fertility ≈ 4.5-5.5)
```


**باستخدام Tokenizer عربي مخصص** (مُدرَّب بـ 50K مفردات على كوربص عربي ضخم):


```text
تقدير: ~10 كلمات → ~12-14 رمزًا (Fertility ≈ 1.2-1.4)
```


**الفرق الكمي**: نفس الجملة بالضبط تستهلك **3-4 أضعاف** عدد الرموز عند استخدام tokenizer غير مُحسَّن للعربية! هذا يعني أن نافذة السياق (Context Window) المتاحة فعليًا للمحتوى العربي تنكمش بمقدار 3-4 مرات، وتكلفة الحوسبة (التي تتناسب تقريبًا خطيًا مع طول التسلسل في بعض الأجزاء، وتربيعيًا في Attention التقليدي) ترتفع بشكل كبير لنفس المحتوى الدلالي بالضبط.


In [ ]:
def compare_tokenizer_fertility_demo():
    """
    محاكاة مبسطة لتوضيح فجوة Fertility بين tokenizer عام وtokenizer عربي مخصص
    (الأرقام الفعلية تعتمد على بيانات تدريب كل tokenizer، لكن الاتجاه حقيقي
    ومُوثَّق في أدبيات المجال، مثل تقرير Jais وتقرير AraGPT2).
    """
    sentence = "تساهم نماذج المحوّلات في تطوير تطبيقات معالجة اللغات الطبيعية بشكل كبير"
    num_words = len(sentence.split())

    # قيم تقديرية واقعية من الأدبيات المنشورة لمقارنات مشابهة
    scenarios = {
        "GPT-2 Tokenizer (إنجليزي أساسًا, 50K مفردات)": 52,
        "GPT-4 Tokenizer (cl100k_base, متعدد اللغات, 100K)": 28,
        "Tokenizer عربي مخصص (50K مفردات مُدرَّب على كوربص عربي)": 13,
    }

    print(f"الجملة: {sentence}")
    print(f"عدد الكلمات: {num_words}\n")
    print(f"{'Tokenizer':<55}{'عدد Tokens':<12}{'Fertility':<10}")
    print("-" * 77)
    for name, num_tokens in scenarios.items():
        fertility = num_tokens / num_words
        print(f"{name:<55}{num_tokens:<12}{fertility:<10.2f}")

compare_tokenizer_fertility_demo()


**الناتج**:


```text
الجملة: تساهم نماذج المحوّلات في تطوير تطبيقات معالجة اللغات الطبيعية بشكل كبير
عدد الكلمات: 10

Tokenizer                                             عدد Tokens  Fertility
-----------------------------------------------------------------------------
GPT-2 Tokenizer (إنجليزي أساسًا, 50K مفردات)            52          5.20
GPT-4 Tokenizer (cl100k_base, متعدد اللغات, 100K)       28          2.80
Tokenizer عربي مخصص (50K مفردات مُدرَّب على كوربص عربي)   13          1.30
```


هذا الفرق (5.20 مقابل 1.30) يعني عمليًا أن معالجة نفس الكمية من النص العربي بـ Tokenizer غير محسَّن تكلف **4 أضعاف تقريبًا** من حيث طول التسلسل، وبالتالي من حيث حوسبة Attention وذاكرة KV Cache.


## 5. نصائح وحيل

- **درّب Tokenizer على عيّنة تمثيلية فعلاً** من البيانات النهائية (نفس توزيع اللغات والمجالات)، وليس فقط على مصدر واحد (Wikipedia فقط مثلاً)، وإلا سينحاز لأسلوب ذلك المصدر.
- **استخدم Byte-Level BPE** بدلاً من Character-Level BPE البحت، لأنه يضمن عدم وجود رموز "Unknown" إطلاقًا (أي نص قابل للتفكيك إلى بايتات UTF-8 خام كحل أخير).
- عند بناء نموذج متعدد اللغات (عربي+إنجليزي مثلاً)، **وازن نسب كل لغة في بيانات تدريب Tokenizer** لتفادي هيمنة لغة واحدة على المفردات المُتعلَّمة على حساب الأخرى.
- أضف **رموزًا خاصة (Special Tokens)** مسبقًا أثناء تدريب Tokenizer (مثل `<|user|>`, `<|assistant|>`) حتى لو لم تبدأ بعد مرحلة SFT، لتفادي الحاجة لإعادة التدريب أو لتوسيع المفردات لاحقًا بطريقة مرتجلة.


## 6. أخطاء شائعة ومصائد

| الخطأ | النتيجة | الحل |
|---|---|---|
| استخدام tokenizer إنجليزي جاهز لمشروع عربي | Fertility مرتفع جدًا، هدر سياق وحوسبة | تدريب tokenizer مخصص على كوربص عربي |
| عدم تطبيق NFC قبل التدريب | تشتت المفردات (نفس الحرف يُمثَّل برمزين مختلفين) | تطبيق `unicodedata.normalize("NFC", text)` دائمًا كخطوة أولى |
| اختيار حجم مفردات عشوائي بدون تحليل | إما إهدار معاملات (مفردات أكبر من اللازم) أو Fertility مرتفع (أصغر من اللازم) | قياس Fertility على عدة أحجام مرشحة واختيار أفضل توازن |
| تغيير Tokenizer بعد بدء التدريب | يُفسد كل التدريب السابق (المفردات/الفهرسة مختلفة تمامًا) | تثبيت Tokenizer نهائيًا **قبل** بدء تدريب النموذج الفعلي |


## 7. جدول مقارنة

| الخوارزمية | التعامل مع الصرف العربي الغني | سرعة الترميز | الاستخدام في نماذج حقيقية |
|---|---|---|---|
| **Byte-Level BPE** | جيد (يتعلم الجذور/اللواحق الشائعة تلقائيًا من البيانات) | سريعة جدًا | GPT، LLaMA، معظم LLMs الحديثة |
| **WordPiece** | مشابه لـ BPE لكن بمعيار اختيار دمج مختلف (احتمالية بدل تكرار) | سريعة | BERT ومشتقاته |
| **Unigram (SentencePiece)** | أفضل نظريًا في التعامل مع التجزئة المتعددة المحتملة لكلمة واحدة | متوسطة | T5، XLNet، ALBERT |
| **Morphological Tokenization** (تجزئة صرفية صريحة، مثل Farasa/CAMeL) | ممتاز (يفهم فعليًا الجذر والوزن الصرفي) | أبطأ، يتطلب أدوات لغوية متخصصة | أبحاث متخصصة، ليس شائعًا في LLMs الحديثة واسعة النطاق |


## 8. ملخص أفضل الممارسات

- طبّق تطبيع NFC دائمًا وبثبات على كل بيانات التدريب (Tokenizer والنموذج) قبل أي معالجة أخرى.
- درّب Tokenizer على عيّنة ضخمة وتمثيلية من التوزيع النهائي المستهدف للبيانات.
- قِس Fertility دائمًا كمعيار كمي للمقارنة بين إعدادات مختلفة، ولا تعتمد على الحدس فقط.
- وازن حجم المفردات بين الكفاءة (Fertility منخفض) وتكلفة معاملات Embedding/Output.
- ثبّت قرار Tokenizer نهائيًا قبل بدء تدريب النموذج الفعلي — لا رجعة بعد ذلك دون إعادة تدريب كاملة.
- أضف الرموز الخاصة اللازمة لمراحل لاحقة (SFT، أدوات/Agents) من البداية.


## 9. أسئلة مقابلات

**سؤال 1**: ما هو Fertility وكيف يؤثر مباشرة على تكلفة تدريب وتشغيل LLM؟

> **الإجابة النموذجية**: Fertility هو متوسط عدد الـ Tokens الناتجة لكل كلمة في لغة معينة عند استخدام tokenizer محدد. ارتفاعه يعني أن نفس المحتوى الدلالي يتطلب تسلسل رموز أطول. بما أن تكلفة Attention تتناسب تربيعيًا تقريبًا مع طول التسلسل (في Attention التقليدي)، وبما أن حجم نافذة السياق المتاحة فعليًا (بالكلمات الفعلية) تتقلص مع ارتفاع Fertility، فإن ارتفاعه يعني: تكلفة حوسبة أعلى، ذاكرة KV Cache أكبر، وقدرة أقل على معالجة نصوص طويلة فعليًا ضمن نفس حد الرموز الأقصى (Context Length) للنموذج.

**سؤال 2**: لماذا يجب تثبيت Tokenizer قبل بدء تدريب النموذج، ولماذا لا يمكن تغييره لاحقًا بسهولة؟

> **الإجابة النموذجية**: طبقة Embedding للنموذج (وطبقة Output Head إن لم تكن مربوطة Weight-Tied) تُبنى بحجم ثابت $V \times d_{model}$ حيث V هو حجم مفردات Tokenizer المحدَّد. كل معامل من هذه الطبقة "يتعلم" تمثيلاً مرتبطًا برقم تعريف (ID) معين يقابل رمزًا معينًا في ذلك Tokenizer تحديدًا. إذا غيّرت Tokenizer بعد بدء التدريب، فإن الفهرسة (أي رقم يقابل أي رمز) تتغير بالكامل، مما يجعل كل ما تعلمه النموذج في طبقة Embedding عديم المعنى فجأة (الرقم 523 الذي كان يمثل كلمة "القط" قد يمثل الآن كلمة مختلفة تمامًا). هذا يتطلب عمليًا إعادة تدريب طبقة Embedding/Output من الصفر على الأقل، إن لم يكن النموذج بأكمله.

**سؤال 3**: لماذا نستخدم Byte-Level BPE تحديدًا بدلاً من Character-Level BPE في أغلب LLMs الحديثة؟

> **الإجابة النموذجية**: Character-Level BPE يتعامل مع "الأحرف" كوحدة أساسية، لكن هذا يتطلب معرفة مسبقة بكل الأحرف المحتملة في كل اللغات المستهدفة، وأي حرف نادر غير موجود في بيانات التدريب (رموز Emoji، أحرف لغات أخرى، رموز خاصة) سيُصنَّف كـ"Unknown Token" (`<UNK>`) ويفقد النموذج أي معلومة عنه. Byte-Level BPE يعمل على مستوى البايتات الخام لترميز UTF-8 (256 قيمة ممكنة فقط)، وبما أن **أي** نص في **أي** لغة يمكن تمثيله كتسلسل بايتات UTF-8 صالح، فإن هذا يضمن رياضيًا عدم وجود أي رمز "Unknown" إطلاقًا — حتى النص الذي لم يُرَ إطلاقًا أثناء تدريب Tokenizer يمكن دائمًا تفكيكه لبايتاته الأساسية كحل أخير.


## 10. تمرين عملي

درِّب Tokenizer بـ BPE على كوربص عربي صغير (استخدم فصل 18 لتنظيف عينة من نصوص عربية متاحة لك، مثل مقالات ويكيبيديا)، بحجمي مفردات مختلفين (8K و32K)، ثم قس Fertility لكل منهما على نفس مجموعة اختبار من 20 جملة متنوعة. قارن النتائج وناقش أي حجم مفردات يُناسب حجم الكوربص المتاح لديك (تلميح: المفردات الكبيرة جدًا على كوربص صغير جدًا تُنتج "Over-fitting" في المفردات نفسها — رموز نادرة جدًا الاستخدام).


## 11. مراجع للتعمّق

- Sennrich et al. (2016), *"Neural Machine Translation of Rare Words with Subword Units"* — الورقة الأصلية لـ BPE في معالجة اللغة.
- Kudo & Richardson (2018), *"SentencePiece: A simple and language independent subword tokenizer"*.
- Inoue et al. (2021) / Jais Technical Report — مناقشات عملية حول تحسين Tokenizer للعربية تحديدًا.
- وثائق Unicode Standard Annex #15 — لفهم أشكال التطبيع NFC/NFD/NFKC/NFKD بشكل كامل.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — بناء LLM من الصفر / الفصل 19: تدريب Tokenizer مخصص.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
